# W02b.1 — LLM API Basics with LangChain

We'll start with the basics of LLM API calls with LangChain.

The rest of the notebook assumes you've installed the required packages and set up your .env file with the required keys as shown in class.

In [1]:
# import the API keys
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 1. Your first call

LangChain provides a small number of core features to get started with AI agents: `chat_models`, `messages`, `tools`, `agents`.

`init_chat_model` is the universal constructor for a chat model: give it a model name, get back a model with the same interface regardless of provider.

### Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
claude = init_chat_model(model="claude-sonnet-4-6")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

### Sending an inference request

Use `invoke` to send a request to the model.

In [4]:
question = "In one sentence: What does temperature do in an API call?"

In [5]:
#TODO: send the request and store the response
#TODO: display the response

response = model.invoke(question)
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': "Temperature in an API call controls the randomness and creativity of the model's responses, where lower values yield more predictable and deterministic text, and higher values produce more varied and imaginative outputs.", 'extras': {'signature': 'El4KXAFpFH0TbK2Gh6FnkhNqv47WlWjJgO5VM4LgSR4xTsWix8EgZ4QJkh4oH7QSbjwnx0caAUHF7ySmrSFZyXltEZEzhVGb1IsA8neLJHups2YfLlU36Kn1T7QBz4um'}}]


`invoke` returned an `AIMessage` object, not a string. It carries the text plus metadata about how it was produced. Look at what else is in there.

In [6]:
from pprint import pprint

print(type(response).__name__)
pprint(response.response_metadata)

AIMessage
{'finish_reason': 'STOP',
 'model_name': 'gemini-3.5-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


## 2. Messages and roles

APIs take a list of messages, each with a role:

| Role | Class | Who it is |
|---|---|---|
| system | `SystemMessage` | You, the developer: persona, rules |
| user | `HumanMessage` | The end user |
| assistant | `AIMessage` | The model's own past turns |

The system prompt is used to steer the assistant persona.

Example: `SystemMessage(content="your message")`

In [7]:
system_prompt = "You are a teaching assistant for an Agentic AI course. Answer in one sentence."
user_question = "What does temperature do in an API call?"

In [8]:
from langchain.messages import SystemMessage, HumanMessage

#TODO: define the list of messages containing system and user prompts.
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_question),
]

response = model.invoke(messages)
print(response.content)

[{'type': 'text', 'text': "Temperature controls the randomness and creativity of the model's output by scaling the logits before sampling, where a higher value produces more diverse and unpredictable responses while a lower value makes the output more deterministic and focused.", 'extras': {'signature': 'El4KXAFpFH0T4dI99mmZ6XH8p0pKaqwaoghYELsLaWiliKL8ZxWAsUSLxtz95khB/5YIlu38epW+XEzIwrwnnx6yRa2zEUXIYAztvXyg/XXlT5M9Kqo0QSi7YP5IWXf2'}}]


## 3. The model is stateless

The model keeps no memory between calls. If you want a conversation, you must send the whole history every time. 

Let's see this in action. First turn:

In [9]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage(content="You are a concise tutor."),
    HumanMessage(content="Hello from Boston. Give me a two-line analogy for what an embedding is."),
]

first_response = model.invoke(conversation)


In [10]:
type(first_response)

langchain_core.messages.ai.AIMessage

In [11]:
first_response

AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like a GPS coordinate for meaning, placing related words close together on a map. \nJust as latitude and longitude represent physical locations, numbers represent semantic concepts so computers can understand context.', 'extras': {'signature': 'El4KXAFpFH0TxG9WuKHXLlZrQBKV38oMkR+0MV0xO1wJKtlnUuyEUzCotJ+6QmCtd31IOw2NL2kBeCiyoCb5hsL1xWGE6t9xragZpSLVDTDbwglWCWwt48gB+EuPvV89'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b63d-36ea-7862-b53f-a40305623d61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 46, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}})

In [12]:
next_message = HumanMessage(content="Where do I live?")
next_response = model.invoke([next_message])
print(next_response.content)

[{'type': 'text', 'text': "I don't know where you live. I don't have access to your personal information or location data unless you share it with me!", 'extras': {'signature': 'El4KXAFpFH0T9hozhxj86fqNic8uom5JS5MYe6H1SrDEv9iEw24XKobvGWZ35xUGFsfBdYWSI5JjLhE1957M05KUTk1O7V5qFyHaAHt5xyVKNE5AXf47hKS72S/4GhgC'}}]


In [13]:
print(first_response.content)

[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like a GPS coordinate for meaning, placing related words close together on a map. \nJust as latitude and longitude represent physical locations, numbers represent semantic concepts so computers can understand context.', 'extras': {'signature': 'El4KXAFpFH0TxG9WuKHXLlZrQBKV38oMkR+0MV0xO1wJKtlnUuyEUzCotJ+6QmCtd31IOw2NL2kBeCiyoCb5hsL1xWGE6t9xragZpSLVDTDbwglWCWwt48gB+EuPvV89'}}]


To help the model remember previous conversations, we need to append the model's reply to the history:

In [14]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={})]


In [15]:
conversation.append(first_response)

In [16]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like a GPS coordinate for meaning, placing related words close together on a map. \nJust as latitude and longitude represent physical locations, numbers represent semantic concepts so computers can understand context.', 'extras': {'signature': 'El4KXAFpFH0TxG9WuKHXLlZrQBKV38oMkR+0MV0xO1wJKtlnUuyEUzCotJ+6QmCtd31IOw2NL2kBeCiyoCb5hsL1xWGE6t9xragZpSLVDTDbwglWCWwt48gB+EuPvV89'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b63d-36ea-7862-b53f-a40305623d61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_t

In [17]:
model.invoke([next_message])

AIMessage(content=[{'type': 'text', 'text': "I don't know where you live. I don't have access to your personal information, location data, or IP address unless you share it with me.", 'extras': {'signature': 'El4KXAFpFH0T4bSRsB0bn/84Qz3gkJdMwI/JZ2f1GzkhOwGgehrzed8o4d9/mybSb2UASqx/v2RuwyaNmewT3L20rG8V/CVWjm4O7ESDAqXYcU95JhNCxPUvRpJPxz0W'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b63d-3da0-7ec2-bb8d-c989feca8b9a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 33, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}})

We need to append the follow-up to the original list of messages and send everything again:

In [18]:
conversation

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like a GPS coordinate for meaning, placing related words close together on a map. \nJust as latitude and longitude represent physical locations, numbers represent semantic concepts so computers can understand context.', 'extras': {'signature': 'El4KXAFpFH0TxG9WuKHXLlZrQBKV38oMkR+0MV0xO1wJKtlnUuyEUzCotJ+6QmCtd31IOw2NL2kBeCiyoCb5hsL1xWGE6t9xragZpSLVDTDbwglWCWwt48gB+EuPvV89'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b63d-36ea-7862-b53f-a40305623d61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_t

In [19]:
conversation.append(next_message)

In [20]:
second_message = model.invoke(conversation)
print(second_message.content)

[{'type': 'text', 'text': 'You live in Boston, Massachusetts, based on your greeting!', 'extras': {'signature': 'El4KXAFpFH0TPlNC2KpAzJkXsBoguVNnaYP51U/YcGIbAd1AT5TjFPYqODOVzAEVjQW9dTOUhF8DMOKGpUYxcrZsm3MisAeBYf3uSbhmKofO6oJjkTvCllT+vJyAwghJ'}}]


In [21]:
new_message = HumanMessage("Give me another analogy that's more relatable to someone who loves cooking.")
conversation.append(new_message)

In [22]:
last_response = model.invoke(conversation)

print(last_response.content)

[{'type': 'text', 'text': "*Boston! (Unless you've moved since your first message.)*\n\nAn embedding is like the flavor profile of a dish, where ingredients with similar tastes sit close together on a spice rack. \nJust as cinnamon and nutmeg share a neighborhood of warm spices, words with similar meanings occupy nearby coordinates in a recipe space.", 'extras': {'signature': 'El4KXAFpFH0TN0WQoE6pqvMFiiRQDy/Wj6ghdStzFemdcSTnznsa02wexTrsMVUp55BcUCwn5Ul1UX7rRjBP2qkfeyGepOEBFNaZwYFb8Flty5lCDIIwoRZQo7IuW2e0'}}]


In [23]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! \n\nAn embedding is like a GPS coordinate for meaning, placing related words close together on a map. \nJust as latitude and longitude represent physical locations, numbers represent semantic concepts so computers can understand context.', 'extras': {'signature': 'El4KXAFpFH0TxG9WuKHXLlZrQBKV38oMkR+0MV0xO1wJKtlnUuyEUzCotJ+6QmCtd31IOw2NL2kBeCiyoCb5hsL1xWGE6t9xragZpSLVDTDbwglWCWwt48gB+EuPvV89'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b63d-36ea-7862-b53f-a40305623d61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_t

Every extra turn grows the context the model re-reads. This is why long agent sessions get expensive.

## 4. The sampling knobs

As we discussed, the model outputs a probability distribution over the next token. The API knobs decide how we draw from it.

- `temperature` divides the logits before softmax. Low sharpens (more deterministic), high flattens.
- `top_p` keeps only the smallest set of tokens covering probability mass p.
- `max_tokens` is a hard cap on output length.

BUT: reasoning-first models (the `gpt-5` family) fix their sampling internally. For these experiments we'll use a standard chat model without reasoning.

In [24]:
sampler = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
#sampler = ChatOllama(model="qwen3.5:4b", reasoning=False)
#sampler = init_chat_model(model="gpt-4.1-mini")


Sampling is random, so a single call proves nothing. A small helper to run the same prompt several times:

In [25]:
def sample_n(prompt, n=4, **model_kwargs):
    #m = init_chat_model(model="gpt-4.1-mini", **model_kwargs)
    m = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", **model_kwargs)
    return [m.invoke(prompt).text.strip() for _ in range(n)]

prompt = "Invent a name for Northeastern University's mascot, husky. Reply with the name only."


In [26]:
print("temperature = 0.0  (near-greedy)")

for out in sample_n(prompt, temperature=0.0):
    print(" ", out)

temperature = 0.0  (near-greedy)


/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/si

  Paws
  Paws
  Paws
  Paws


In [27]:
print("temperature = 1.4  (flattened distribution)")
for out in sample_n(prompt, temperature=1.4):
    print(" ", out)

temperature = 1.4  (flattened distribution)


/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/si

  Paws
  Paws
  Paws
  Pawsibility


Near-identical names at 0.0, real variety at 1.4. Two caveats: even at temperature 0 the answers can occasionally differ, and high temperature is not more intelligence, only a flatter distribution.

In [28]:
print("top_p = 0.1  (only the most probable tokens survive)")
for out in sample_n(prompt, top_p=0.1):
    print(" ", out)

top_p = 0.1  (only the most probable tokens survive)


/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) top_p will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) top_p will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) top_p will be ignored.
  request = self._build_request_config(
/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/langch

  Paws
  Paws
  Pawsibility
  Paws


### Choosing settings:

| Task | Setting |
|---|---|
| Tool calls, structured output | temperature 0 to 0.3 |
| Factual Q&A, extraction | low 0.3 to 0.5 |
| Brainstorming, drafts | 0.8 to 1.2, several samples |
| General chat | defaults; tune one knob, not both |

**Rule of thumb for agents: agents loop, and a small weirdness rate per step compounds across twenty steps. Keep agent temperature low.**

### max_tokens
It's also possible to limit the number of tokens to be generated. Use `max_tokens` as a cost ceiling. 

In [29]:
#capped = ChatOllama(model="qwen3.5:4b", reasoning=False, max_tokens = 50)
capped = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_tokens=50)
r = capped.invoke("Explain how backpropagation works.")


In [30]:
print(r.content)
print("\nstop reason:", r.response_metadata.get("finish_reason") or r.response_metadata.get("stop_reason"))

[{'type': 'text', 'text': 'At its core, **backpropagation** (short for "backward propagation of errors") is the central mathematical mechanism that allows artificial neural networks to learn. \n\nIf a neural network is like a student taking a test, **forward', 'extras': {'signature': 'El4KXAFpFH0TqHsNBcN8tPvjV1rFbqRsNj9RyiKxZlWduMSvQWvFNimd51/GXJJs8/selsIfHh+Ap0S5XZENfkasSkCXAcNIjLPsTO1xDaoCCeX54KNgcCIjr2vEt2Gq'}}]

stop reason: MAX_TOKENS


The text stops mid-sentence and the stop reason says `length`, not `stop`. 

## 5. Streaming

Generation is one token per forward pass. Streaming exposes that rhythm instead of making you wait for the full reply.

Use `model.stream()` to stream each chunk as you receive them. Will need to iterate through the 
- model.stream("your query") will return the response as a sequence

In [31]:
#TODO: iterate through the stream
#TODO: display the chunks
for chunk in model.stream("What color is sky?"):
    print(chunk.text, end="|", flush=True)



The sky| is usually **blue** during a clear day. 

However, the color changes depending on the time| of day and the weather:
* **Sunrise and Sunset:** Red, orange, pink, and purple.
* **Night|:** Black or very dark blue.
* **Cloudy or Stormy:** White, gray, or even dark blue-|black. 

The sky appears blue during the day because of a phenomenon called **Rayleigh scattering**. The Earth's atmosphere scatters| sunlight in all directions, and because blue light travels in smaller, shorter waves than other colors, it gets scattered more than the| other colors, making the sky look blue to our eyes.|||